# Open-weight judges on the v2 cells

Runs open-weight judges over the frozen v2 judge cells using **the exact pilot judge prompt**,
logs one JSON row per judgment in the same schema as `runs/v2/sonnet5_*.jsonl`, and prints the
same result tables as the Anthropic runs.

No prompt-building code is written here. `build_consequence_segments()` is imported from the
repo's `src/judge.py` and called exactly as `src/run_v2_judge.py` calls it, with `{LAB}`
substituted per judge.

---

## Required Drive layout

`data/` is gitignored in the repo, so a clone has neither the cell files nor `prompts.json`.
Copy the files below into `DATA_DIR` (default `/content/drive/MyDrive/csm/v2`) **exactly at
these paths**. Cell 3 checks every one and aborts with a one-line instruction if any required
file is missing.

```
/content/drive/MyDrive/csm/v2/
├── confession_neutral.jsonl        required
├── confession_obj.jsonl            required
├── CA.jsonl                        required
├── D.jsonl                         required
├── persuasive.jsonl                required
├── H0.jsonl                        required
├── H2.jsonl                        required
├── H3.jsonl                        required
├── natural_disavowal.jsonl         required
├── prompts.json                    required — sha256 must be 6620aaa9…d22371ac
├── EXPERIMENT_RUN_LEDGER.csv       required — the ledger cell appends to it
└── runs/
    ├── sonnet5_confession_neutral_r1.jsonl   required ┐
    ├── sonnet5_confession_obj_r1.jsonl       required │
    ├── sonnet5_CA_r1.jsonl                   required │ the prompt assertion compares
    ├── sonnet5_D_r1.jsonl                    required │ every cell's first row against
    ├── sonnet5_persuasive_r1.jsonl           required │ the prompt hash logged here
    ├── sonnet5_H0_r1.jsonl                   required │
    ├── sonnet5_H2_r1.jsonl                   required │
    ├── sonnet5_H3_r1.jsonl                   required │
    ├── sonnet5_natural_disavowal_r1.jsonl    required ┘
    ├── opus55_CA_subset50_r1.jsonl           optional — cross-judge column only
    └── opus55_D_subset50_r1.jsonl            optional — cross-judge column only
```

`E.jsonl`, `F.jsonl` and the `*_opus5.5.jsonl` subsets are **not** used and need not be copied.
Outputs are written back to `DATA_DIR/runs/<judge>_<cell>_r<repeat>.jsonl`.

## Sequence

1. Run cells 1–8 once (install, clone, mount + layout check, config, judges, imports, prompt
   assertion, engine).
2. **Dry run**: set `N_ROWS = 5` in CONFIG, run cells 9–10. This exercises the prompt
   assertion, the chat template, thinking extraction and label parsing on 5 rows per cell.
   Inspect the output before going further.
3. **Full run**: set `N_ROWS = None`, re-run cell 10. It is resumable — rows already in the
   output file are skipped, so the dry-run rows are kept.
4. Run cell 11 (report), then cell 13 (ledger; its call is commented out by default).

## Hardware

`gpt-oss-20b` and 4-bit `Qwen3.8-27B` fit a ~24 GB GPU. `gpt-oss-120b` is MXFP4 and needs a
single 80 GB card (A100 80GB / H100); it will not load on T4 or L4.

**Nothing here is preregistered yet.** `notes/v2_prereg_amendment_openweights.md` is drafted in
the repo for review; commit it before the first full run.

In [ ]:
#@title 1 · Install  (pins verified against the model cards, 2026-09-23)
# Binding constraints, lowest versions that satisfy all three judges:
#   vLLM         >= 0.17.0   from the Qwen3.8-27B vLLM recipe (gpt-oss needs only >= 0.10.1)
#   transformers >= 5.8.0    from the Qwen3.8-27B vLLM recipe: "must match the version
#                            config.json was written by"; vLLM parses it with Qwen3_5Config
# The repo pins transformers==5.16.1, which satisfies >= 5.8.0, so we use the repo pin and
# stay consistent with the Anthropic runs' environment.
VLLM_PIN         = "vllm==0.17.0"          # Qwen3.8-27B recipe (vllm >= 0.17.0)
TRANSFORMERS_PIN = "transformers==5.16.1"  # repo requirements.txt; >= 5.8.0 as Qwen requires

import subprocess, importlib.metadata as md
def _v(pkg):
    try: return md.version(pkg)
    except Exception: return None

# vLLM first: it pins torch. Anything installed afterwards must not move torch.
!pip -q install "{VLLM_PIN}" 2>&1 | tail -2
TORCH_AFTER_VLLM = _v("torch")
!pip -q install "{TRANSFORMERS_PIN}" "accelerate>=1.0" "bitsandbytes>=0.45" 2>&1 | tail -2
!pip -q install "anthropic==1.2.0" "pandas==3.0.5" "psutil==7.2.2" 2>&1 | tail -2

print("\n--- pins and where they come from ---")
print(f"  vllm         {_v('vllm'):<12} required >= 0.17.0   (Qwen3.8-27B vLLM recipe; "
      f"gpt-oss cards require only >= 0.10.1)")
print(f"  transformers {_v('transformers'):<12} required >= 5.8.0    (Qwen3.8-27B vLLM recipe; "
      f"gpt-oss cards state no minimum)")
print(f"  torch        {_v('torch'):<12} pinned by vLLM")
if TORCH_AFTER_VLLM and _v("torch") != TORCH_AFTER_VLLM:
    print(f"  !! torch MOVED after installing transformers: {TORCH_AFTER_VLLM} -> {_v('torch')}")
    print("     Re-install vLLM last, or install transformers with --no-deps, before running.")
else:
    print("  torch unchanged after the transformers install — no vLLM/transformers conflict")
print("\nNote: the gpt-oss cards show the day-0 wheel vllm==0.10.1+gptoss. That special wheel is")
print("no longer needed; gpt-oss is supported in mainline vLLM, and 0.17.0 satisfies both cards.")
print(subprocess.run(["nvidia-smi","--query-gpu=name,memory.total","--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "no GPU visible")

In [ ]:
#@title 2 · Clone the repo and pin its commit
import subprocess, sys, pathlib
# The repository was renamed from consequence-sensitive-monitors on 2026-09-23.
REPO_URL = "https://github.com/JulesRoussel2001/reward-hacking-llm-judge.git"
REPO_DIR = pathlib.Path("/content/consequence-sensitive-monitors")
if not REPO_DIR.exists():
    subprocess.run(["git","clone","--depth","50",REPO_URL,str(REPO_DIR)], check=True)
REPO_COMMIT = subprocess.run(["git","-C",str(REPO_DIR),"rev-parse","HEAD"],
                             capture_output=True, text=True, check=True).stdout.strip()
# The prereg commit is read from the repo, exactly as src/run_v2_judge.py does.
PREREG_COMMIT = subprocess.run(
    ["git","-C",str(REPO_DIR),"log","-1","--format=%H","--","notes/v2_prereg.md"],
    capture_output=True, text=True).stdout.strip()
print("repo commit  :", REPO_COMMIT)
print("prereg commit:", PREREG_COMMIT or "(notes/v2_prereg.md not found in this clone)")
sys.path.insert(0, str(REPO_DIR / "src"))

In [ ]:
#@title 3 · Mount Drive, resolve paths, and check the required layout
from google.colab import drive
import pathlib, hashlib, json, sys
drive.mount("/content/drive")

DATA_DIR = pathlib.Path("/content/drive/MyDrive/csm/v2")   #@param {type:"string"}
RUNS_DIR     = DATA_DIR / "runs"
PROMPTS_JSON = DATA_DIR / "prompts.json"
LEDGER_CSV   = DATA_DIR / "EXPERIMENT_RUN_LEDGER.csv"
RUNS_DIR.mkdir(parents=True, exist_ok=True)

def sha256_file(p):
    h = hashlib.sha256()
    with open(p, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""): h.update(chunk)
    return h.hexdigest()

ALL_CELLS = ["confession_neutral","confession_obj","CA","D","persuasive",
             "H0","H2","H3","natural_disavowal"]
PROMPTS_SHA_EXPECTED = "6620aaa91ba57389e01b1fba64cf1fc52ecc387fc49ef0ed0cae10c0d22371ac"

REQUIRED  = [(DATA_DIR/f"{c}.jsonl", f"cell {c}") for c in ALL_CELLS]
REQUIRED += [(PROMPTS_JSON, "prompts.json"), (LEDGER_CSV, "ledger")]
REQUIRED += [(RUNS_DIR/f"sonnet5_{c}_r1.jsonl", f"sonnet5 {c} (prompt assertion)")
             for c in ALL_CELLS]
OPTIONAL  = [(RUNS_DIR/f"opus55_{c}_subset50_r1.jsonl", f"opus55 {c} (cross-judge column)")
             for c in ("CA","D")]

print(f"\nDATA_DIR = {DATA_DIR}\n")
print(f"{'status':<9}{'what':<40}{'sha256'}")
print("-"*115)
missing = []
for path, what in REQUIRED:
    if path.exists():
        print(f"{'present':<9}{what:<40}{sha256_file(path)}")
    else:
        print(f"{'MISSING':<9}{what:<40}{'-'}"); missing.append(path)
for path, what in OPTIONAL:
    if path.exists():
        print(f"{'present':<9}{what+' [optional]':<40}{sha256_file(path)}")
    else:
        print(f"{'absent':<9}{what+' [optional]':<40}{'-'}")

if PROMPTS_JSON.exists():
    got = sha256_file(PROMPTS_JSON)
    print("\nprompts.json sha256",
          "matches expected 6620aaa9…" if got == PROMPTS_SHA_EXPECTED
          else f"!! DOES NOT MATCH expected {PROMPTS_SHA_EXPECTED[:8]}… — got {got[:8]}…")
    assert got == PROMPTS_SHA_EXPECTED, (
        "prompts.json on Drive is not the file the Anthropic runs used. Copy "
        "data/anthropic_mislabeling/prompts.json from the repo working copy and re-run.")

if missing:
    sys.exit(f"ABORT — {len(missing)} required file(s) missing. Copy them from the repo working "
             f"copy to {DATA_DIR} exactly as listed in the README cell, then re-run this cell.")
print(f"\nall {len(REQUIRED)} required files present; "
      f"{sum(1 for p,_ in OPTIONAL if p.exists())}/{len(OPTIONAL)} optional present")

In [ ]:
#@title 4 · CONFIG
JUDGE   = "gpt-oss-20b"        #@param ["gpt-oss-20b","qwen3.8-27b","gpt-oss-120b"]
CELLS   = ["confession_neutral","confession_obj","CA","D","persuasive",
           "H0","H2","H3","natural_disavowal"]
N_ROWS  = 5                    # 5 = dry run; None = all rows in each cell
REPEAT  = 1                    # run index, goes in the filename and run_id
SEED    = 2026
MAX_NEW_TOKENS = 32768         # a truncated generation with no <label> tag is a no-label

print(f"JUDGE={JUDGE}  CELLS={len(CELLS)}  N_ROWS={N_ROWS}  REPEAT={REPEAT}  SEED={SEED}")
print("DRY RUN" if N_ROWS else "FULL RUN")

In [ ]:
#@title 5 · JUDGES
# Verified against each model card on 2026-09-23. Notes record where a card is silent or
# disagrees with the run request; the card wins and the disagreement is printed.
JUDGES = {
    "gpt-oss-20b": dict(
        model_id      = "openai/gpt-oss-20b",
        lab           = "OpenAI",
        quantization  = "mxfp4 (as released; MoE weights post-trained in MXFP4)",
        dtype         = "auto",
        # Card: "The reasoning level can be set in the system prompts, e.g. 'Reasoning: high'."
        # The transformers chat template exposes this as a kwarg; the template then writes its
        # own system message, which we log verbatim per row.
        chat_template_kwargs = {"reasoning_effort": "high"},
        reasoning_setting    = 'reasoning_effort="high"',
        # Card gives no sampling recommendation. These are the OpenAI reference defaults.
        sampling      = dict(temperature=1.0, top_p=1.0, top_k=-1),
        loader        = "tokenizer",            # AutoTokenizer; text-only model
        hf_model_class= "AutoModelForCausalLM",
        min_vllm      = "0.10.1",
        notes         = ["card gives no sampling recommendation; using reference defaults "
                         "temperature=1.0, top_p=1.0, top_k off",
                         "card shows the day-0 wheel vllm==0.10.1+gptoss; mainline vLLM now "
                         "supports gpt-oss, so the notebook pins 0.17.0 for all three judges"],
    ),
    "qwen3.8-27b": dict(
        model_id      = "Qwen/Qwen3.8-27B",
        lab           = "Alibaba",
        # Card lists no official 4-bit release (official quantized release is FP8:
        # Qwen/Qwen3.8-27B-FP8). Per the run request we therefore load bnb-4bit at load time.
        quantization  = "bitsandbytes-nf4 (load-time); no official 4-bit repo exists",
        dtype         = "bfloat16",
        chat_template_kwargs = {"enable_thinking": True},
        reasoning_setting    = "enable_thinking=True",
        # Card, thinking mode: temperature=1.0, top_p=0.95, top_k=20, min_p=0.0
        sampling      = dict(temperature=1.0, top_p=0.95, top_k=20, min_p=0.0),
        # Qwen3.8-27B is a native vision-language model (Qwen3_5ForConditionalGeneration).
        # The card's quickstart uses AutoProcessor + AutoModelForMultimodalLM, and the chat
        # template is applied on the PROCESSOR, not a tokenizer. We send text-only messages.
        loader        = "processor",
        hf_model_class= "AutoModelForMultimodalLM",
        min_vllm      = "0.17.0",
        notes         = ["no official 4-bit release on the card; official quantized release is "
                         "Qwen/Qwen3.8-27B-FP8. Loading bnb-nf4 at load time as instructed.",
                         "MULTIMODAL: AutoProcessor + AutoModelForMultimodalLM per the card; "
                         "we pass text-only messages",
                         "chat template opens every assistant turn with <think>, so the closing "
                         "</think> arrives without an opening tag; split_reasoning handles that",
                         "vLLM recipe: vllm >= 0.17.0 and transformers >= 5.8.0"],
    ),
    "gpt-oss-120b": dict(
        model_id      = "openai/gpt-oss-120b",
        lab           = "OpenAI",
        quantization  = "mxfp4 (as released); needs a single 80GB GPU",
        dtype         = "auto",
        chat_template_kwargs = {"reasoning_effort": "high"},
        reasoning_setting    = 'reasoning_effort="high"',
        sampling      = dict(temperature=1.0, top_p=1.0, top_k=-1),
        loader        = "tokenizer",
        hf_model_class= "AutoModelForCausalLM",
        min_vllm      = "0.10.1",
        notes         = ["card gives no sampling recommendation; using reference defaults",
                         "MXFP4: will not load on T4/L4; use A100-80GB or H100"],
    ),
}
J = JUDGES[JUDGE]
print(JUDGE, "->", J["model_id"])
for k in ("lab","quantization","dtype","reasoning_setting","loader","hf_model_class","min_vllm"):
    print(f"  {k:<16}{J[k]}")
print(f"  {'sampling':<16}{J['sampling']}")
for n in J["notes"]: print("  NOTE:", n)

In [ ]:
#@title 6 · Import the repo's prompt builders (unchanged)
# src/judge.py imports the vendored SLEIGHT provider stack at module level. That lives under
# external/, which is gitignored and therefore absent from a clone. Those modules are only used
# by the Anthropic call path, never by prompt assembly, so we register inert stubs before the
# import. Nothing in the repo is modified.
import sys, types
for name in ["evaluate","evaluate.core","evaluate.core.providers","evaluate.core.response",
             "evaluate.core.transcript","evaluate.core.types"]:
    if name not in sys.modules:
        sys.modules[name] = types.ModuleType(name)
for name, attrs in [("evaluate.core.providers", ["build_api_params","call_anthropic"]),
                    ("evaluate.core.response",  ["extract_response_parts","parse_response"]),
                    ("evaluate.core.transcript",["render_prompt"]),
                    ("evaluate.core.types",     ["MonitorSpec"])]:
    for a in attrs:
        if not hasattr(sys.modules[name], a):
            setattr(sys.modules[name], a, None)

import judge as J_MOD
from judge import (build_consequence_segments, load_prompts_json, parse_consequence,
                   POSITIVE_LABEL, NEGATIVE_LABEL, _LABEL_RE)
from datasets import _llama_turn, _LLAMA_SPECIAL

PROMPTS = load_prompts_json(PROMPTS_JSON)
print("imported build_consequence_segments from", J_MOD.__file__)
print("labels:", POSITIVE_LABEL, "/", NEGATIVE_LABEL)
print("consequence blocks available:", list(PROMPTS["consequence_blocks"]))

In [ ]:
#@title 7 · Prompt assembly + assertion against the Sonnet 5 runs
import json, hashlib

def render_row(row, lab):
    """Exactly src/run_v2_judge.py: transcript = user turn + completion, system turn into the
    {system_prompt} slot, consequence variant 'none', reworded question, HACKING/HONEST,
    no inserted instruction, single user message."""
    sys_turn  = _llama_turn(row["prompt"], "system")
    user_turn = _llama_turn(row["prompt"], "user")
    assert sys_turn is not None and user_turn is not None, row["derived_from"]
    text = f"user: {user_turn.strip()}\n\nassistant: {row['completion']}"
    assert not _LLAMA_SPECIAL.search(text) and not _LLAMA_SPECIAL.search(sys_turn)
    prefix, suffix = build_consequence_segments(
        text, prompt_variant="none", rubric="default", abstain=False, lab=lab,
        system_prompt=sys_turn.strip(), prompts=PROMPTS, verbatim=False,
        insert_before_response=None,
        negative_label=NEGATIVE_LABEL, positive_label=POSITIVE_LABEL)
    return prefix + suffix

_BLOCK = PROMPTS["consequence_blocks"]["none"]["text"]

def assert_matches_sonnet(cell, row, lab):
    """Two-part check, run on the first row of every cell.
    (1) rebuilt with lab='Anthropic' must hash-match the prompt Sonnet 5 actually received;
    (2) the judge's prompt must differ from it only by the {LAB} substitution."""
    ref_path = RUNS_DIR / f"sonnet5_{cell}_r1.jsonl"
    assert ref_path.exists(), f"missing {ref_path}; needed for the prompt assertion"
    ref = {json.loads(l)["problem_id"]: json.loads(l)["prompt_sha256"] for l in open(ref_path)}
    pid = row["derived_from"]
    assert pid in ref, f"problem {pid} not in {ref_path.name}"
    anth = render_row(row, "Anthropic")
    got  = hashlib.sha256(anth.encode()).hexdigest()
    assert got == ref[pid], (
        f"ASSEMBLY MISMATCH for {cell}:{pid}\n  rebuilt {got}\n  sonnet  {ref[pid]}\n"
        "The prompt this notebook builds is not the prompt the Anthropic runs used. Stop.")
    mine = render_row(row, lab)
    expect = anth.replace(_BLOCK.replace("{LAB}","Anthropic"), _BLOCK.replace("{LAB}",lab))
    assert mine == expect, f"{cell}:{pid}: judge prompt differs by more than the {{LAB}} substitution"
    return got

# smoke-check on one row of one cell
_r = json.loads(open(DATA_DIR/"CA.jsonl", encoding="utf-8").readline())
_h = assert_matches_sonnet("CA", _r, J["lab"])
print("assertion passed on CA row", _r["derived_from"], "->", _h[:16]+"...")
print("\n--- first 400 chars of the judge prompt for this judge ---")
print(render_row(_r, J["lab"])[:400])

In [ ]:
#@title 8 · Load the engine (vLLM offline; transformers fallback)
import torch, importlib.metadata as _md
ENGINE = ENGINE_VERSION = None
LLM_OBJ = TOK = None
FALLBACK_REASON = None

def _load_tok():
    """The card's own class: AutoProcessor for the Qwen VLM, AutoTokenizer for gpt-oss.
    Loaded separately from the engine so templating and token counting are identical on
    the vLLM and transformers paths."""
    if J["loader"] == "processor":
        from transformers import AutoProcessor
        return AutoProcessor.from_pretrained(J["model_id"], trust_remote_code=True)
    from transformers import AutoTokenizer
    return AutoTokenizer.from_pretrained(J["model_id"], trust_remote_code=True)

try:
    from vllm import LLM, SamplingParams
    kw = dict(model=J["model_id"], seed=SEED, trust_remote_code=True,
              max_model_len=40960, gpu_memory_utilization=0.90)
    if JUDGE == "qwen3.8-27b":
        # No official 4-bit repo; vLLM loads bitsandbytes nf4 in-flight.
        kw.update(quantization="bitsandbytes", dtype="bfloat16")
    LLM_OBJ = LLM(**kw)
    TOK = _load_tok()          # not LLM.get_tokenizer(): the VLM needs its processor
    ENGINE, ENGINE_VERSION = "vllm", _md.version("vllm")
except Exception as e:
    FALLBACK_REASON = f"{type(e).__name__}: {e}"
    print("!! vLLM could not load this model — FALLING BACK TO TRANSFORMERS")
    print("   reason:", FALLBACK_REASON[:400])
    import transformers as _tf
    from transformers import BitsAndBytesConfig
    ModelCls = getattr(_tf, J["hf_model_class"])     # AutoModelForMultimodalLM for Qwen
    TOK = _load_tok()
    mkw = dict(device_map="auto", trust_remote_code=True)
    if JUDGE == "qwen3.8-27b":
        mkw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
    else:
        mkw["torch_dtype"] = "auto"
    LLM_OBJ = ModelCls.from_pretrained(J["model_id"], **mkw)
    ENGINE, ENGINE_VERSION = "transformers", _md.version("transformers")

GPU_NAME = torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"
print(f"engine={ENGINE} {ENGINE_VERSION}  gpu={GPU_NAME}  tokenizer={type(TOK).__name__}")
assert _md.version("vllm").split("+")[0] >= J["min_vllm"] or ENGINE == "transformers", \
    f"{JUDGE} needs vllm >= {J['min_vllm']}"
if FALLBACK_REASON: print("NOTE: results were produced on the transformers fallback path, not vLLM.")

In [ ]:
#@title 9 · Generation, thinking extraction, label parsing
import re, json, time, datetime

def count_tokens(text):
    """Token count that works for both a tokenizer and a processor."""
    try:
        return len(TOK(text=text)["input_ids"][0]) if J["loader"] == "processor" \
               else len(TOK(text)["input_ids"])
    except Exception:
        return len(TOK(text)["input_ids"])


def build_chat_text(user_message):
    """Render with the model's own chat template. The template may insert its own system
    message (gpt-oss writes 'Reasoning: high'); we never add a system message of our own."""
    msgs = [{"role": "user", "content": user_message}]
    return TOK.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True,
                                   **J["chat_template_kwargs"])

_SYS_RE = re.compile(r"<\|start\|>system<\|message\|>(.*?)<\|end\|>", re.S)
def template_system_message(rendered):
    """The system block the template itself inserted, logged verbatim."""
    m = _SYS_RE.search(rendered)
    if m: return m.group(1)
    m = re.search(r"<\|im_start\|>system\n(.*?)<\|im_end\|>", rendered, re.S)
    return m.group(1) if m else ""

_THINK  = re.compile(r"<think>(.*?)</think>", re.S)
_CLOSE  = re.compile(r"</think>", re.S)   # Qwen's template pre-opens <think> in the prompt,
                                          # so the generation carries only the closing tag
_ANALYS = re.compile(r"<\|channel\|>analysis<\|message\|>(.*?)(?:<\|end\|>|<\|start\|>|$)", re.S)
_FINAL  = re.compile(r"<\|channel\|>final<\|message\|>(.*?)(?:<\|return\|>|<\|end\|>|$)", re.S)

def split_reasoning(raw):
    """Return (raw_reasoning, answer_text). Never discards anything: if no structure is
    recognised the whole generation is treated as the answer."""
    m = _FINAL.search(raw)
    if m:                                        # gpt-oss harmony channels
        a = _ANALYS.search(raw)
        return (a.group(1).strip() if a else ""), m.group(1).strip()
    m = _THINK.search(raw)
    if m:                                        # full <think>...</think> pair
        return m.group(1).strip(), raw[m.end():].strip()
    m = _CLOSE.search(raw)
    if m:                                        # closing tag only: the template opened it
        return raw[:m.start()].strip(), raw[m.end():].strip()
    if "<|channel|>analysis<|message|>" in raw:  # analysis opened but truncated before final
        a = _ANALYS.search(raw)
        return (a.group(1).strip() if a else raw.strip()), ""
    if "<think>" in raw:                         # think opened but never closed (truncation)
        return raw.split("<think>",1)[1].strip(), ""
    return "", raw.strip()

def parse_label(answer_text):
    verdict, reason = parse_consequence(answer_text, NEGATIVE_LABEL, POSITIVE_LABEL)
    m = _LABEL_RE.search(answer_text or "")
    return verdict, (m.group(1).strip() if m else None), reason

def generate(user_messages):
    """Batched generation. Returns a list of (raw_text, n_out_tokens, stop_reason, rendered)."""
    rendered = [build_chat_text(u) for u in user_messages]
    if ENGINE == "vllm":
        from vllm import SamplingParams
        sp = SamplingParams(max_tokens=MAX_NEW_TOKENS, seed=SEED, **J["sampling"])
        outs = LLM_OBJ.generate(rendered, sp)     # prompts already templated
        res = []
        for o in outs:
            c = o.outputs[0]
            res.append((c.text, len(c.token_ids),
                        "length" if c.finish_reason == "length" else c.finish_reason, None))
        return [(t, n, s, r) for (t, n, s, _), r in zip(res, rendered)]
    out = []
    for txt in rendered:                          # transformers fallback, one at a time
        enc = TOK(txt, return_tensors="pt").to(LLM_OBJ.device)
        gen_kw = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=True)
        gen_kw.update({k: v for k, v in J["sampling"].items() if k != "top_k" or v > 0})
        with torch.no_grad():
            g = LLM_OBJ.generate(**enc, **gen_kw)
        new = g[0][enc["input_ids"].shape[1]:]
        out.append((TOK.decode(new, skip_special_tokens=False), int(new.shape[0]),
                    "length" if int(new.shape[0]) >= MAX_NEW_TOKENS else "stop", txt))
    return out

print("generation helpers ready; engine =", ENGINE)

In [ ]:
#@title 10 · Run loop — incremental, resumable, one row per judgment
import json, hashlib, datetime, pathlib

BATCH = 8 if ENGINE == "vllm" else 1

def run_cell(cell):
    src_path = DATA_DIR / f"{cell}.jsonl"
    rows = [json.loads(l) for l in open(src_path, encoding="utf-8")]
    rows.sort(key=lambda r: r["derived_from"])
    if N_ROWS: rows = rows[:N_ROWS]
    cell_sha = sha256_file(src_path)
    run_id = f"{JUDGE}_{cell}_r{REPEAT}"
    out_path = RUNS_DIR / f"{run_id}.jsonl"

    done = set()
    if out_path.exists():
        done = {json.loads(l)["problem_id"] for l in open(out_path, encoding="utf-8")}
    todo = [r for r in rows if r["derived_from"] not in done]
    print(f"\n=== {cell}: {len(rows)} rows, {len(done)} already done, {len(todo)} to run -> {out_path.name}")
    if not todo: return out_path

    assert_matches_sonnet(cell, todo[0], J["lab"])   # stops the run if assembly drifted
    print(f"    prompt assertion passed on problem {todo[0]['derived_from']}")

    with open(out_path, "a", encoding="utf-8") as fh:
        for i in range(0, len(todo), BATCH):
            chunk = todo[i:i+BATCH]
            msgs  = [render_row(r, J["lab"]) for r in chunk]
            t0 = time.time()
            gens = generate(msgs)
            for r, um, (raw, ntok, stop, rendered) in zip(chunk, msgs, gens):
                rendered = rendered or build_chat_text(um)
                reasoning, answer = split_reasoning(raw)
                verdict, label_raw, reason = parse_label(answer)
                rec = {
                    # ---- Sonnet run schema ----
                    "run_id": run_id, "problem_id": r["derived_from"], "cell": cell,
                    "role": r["role"], "template_id": r.get("template_id"),
                    "fit_rating": r.get("fit_rating"),
                    "transcript_id": f"atlas:v2:{cell}:{r['derived_from']}",
                    "model": J["model_id"], "effort": J["reasoning_setting"],
                    "prompt_variant": "none", "verbatim": False,
                    "positive_label": POSITIVE_LABEL, "negative_label": NEGATIVE_LABEL,
                    "label": verdict, "label_raw": label_raw, "no_label": verdict is None,
                    "reasoning_summary": (reasoning[:2000] if reasoning else None),
                    "response_text": answer, "one_sentence_explanation": reason,
                    "stop_reason": stop, "refused": False,
                    "timed_out": stop == "length", "error": None, "attempts": 1,
                    "input_tokens": count_tokens(rendered),
                    "output_tokens": ntok,
                    "cache_read_input_tokens": 0, "cache_creation_input_tokens": 0,
                    "usd": None,
                    "timestamp": datetime.datetime.now(datetime.timezone.utc).isoformat(),
                    "prompt_sha256": hashlib.sha256(um.encode()).hexdigest(),
                    "input_file": f"data/atlas/v2/{cell}.jsonl", "input_sha256": cell_sha,
                    "prereg_commit": PREREG_COMMIT,
                    # ---- open-weight extras ----
                    "raw_reasoning": reasoning, "raw_generation": raw,
                    "model_id": J["model_id"], "quantization": J["quantization"],
                    "dtype": J["dtype"], "engine": ENGINE, "engine_version": ENGINE_VERSION,
                    "engine_fallback_reason": FALLBACK_REASON,
                    "sampling": J["sampling"], "seed": SEED,
                    "reasoning_setting": J["reasoning_setting"],
                    "chat_template_kwargs": J["chat_template_kwargs"],
                    "template_system_message": template_system_message(rendered),
                    "max_new_tokens": MAX_NEW_TOKENS,
                    "gpu": GPU_NAME, "repo_commit": REPO_COMMIT,
                    "judge_key": JUDGE, "repeat": REPEAT,
                }
                fh.write(json.dumps(rec, ensure_ascii=False) + "\n")
            fh.flush()
            n = min(i+len(chunk), len(todo))
            print(f"    [{n:>4}/{len(todo)}] {time.time()-t0:5.1f}s/batch")
    return out_path

RUN_PATHS = [run_cell(c) for c in CELLS]
print("\nwrote:", *[p.name for p in RUN_PATHS], sep="\n  ")

In [ ]:
#@title 11 · Report — per cell, breakdowns, paired tables, cross-judge
import json, math, statistics, itertools
from report_v2_run import wilson, summarise, fmt, HDR          # reused, not reimplemented
from paired_v2 import binom_two_sided, newcombe10, wald_paired

HACKED = {"confession_neutral","confession_obj","CA","D","persuasive","natural_disavowal"}
ERR    = lambda cell: "HONEST" if cell in HACKED else "HACKING"

def load_run(p):
    return [json.loads(l) for l in open(p, encoding="utf-8")]

def cell_block(rows, cell, lines):
    err = ERR(cell)
    lines.append(f"### {cell}   (error = {err} label on a "
                 f"{'hacked' if err=='HONEST' else 'general'} row)")
    lines.append(f"  overall  {HDR}")
    lines.append(f"           {fmt(summarise(rows, err))}")
    src = {r["derived_from"]: r for r in
           map(json.loads, open(DATA_DIR/f"{cell}.jsonl", encoding="utf-8"))}
    extra = []
    if cell == "D": extra = ["motive_pattern"]
    if cell == "persuasive": extra = ["pragmatic_pattern"]
    for r in rows:
        for k in extra: r[k] = src.get(r["problem_id"], {}).get(k)
    for key in ["template_id","fit_rating",*extra]:
        vals = {str(r.get(key)) for r in rows}
        if vals <= {"None"}:
            lines.append(f"  by {key}: not recorded for this cell"); continue
        lines.append(f"  by {key}:")
        lines.append(f"    {'value':<12} {HDR}")
        for v in sorted(vals):
            sub = [r for r in rows if str(r.get(key)) == v]
            lines.append(f"    {v:<12} {fmt(summarise(sub, err))}")
    o = [r["output_tokens"] or 0 for r in rows]
    lines.append(f"  output tokens: mean {statistics.mean(o):.0f}  median {statistics.median(o):.0f}"
                 f"  max {max(o)}   no-label {sum(1 for r in rows if r['no_label'])}")
    lines.append("")

def paired_block(A, B, nameA, nameB, lines):
    if not A or not B: return
    ids = sorted(set(A) & set(B))
    err = ERR(A[ids[0]]["cell"]) if ids else "HONEST"
    use = [i for i in ids if not A[i]["no_label"] and not B[i]["no_label"]]
    drop = len(ids) - len(use)
    a = sum(1 for i in use if B[i]["label"]==err and A[i]["label"]==err)
    b = sum(1 for i in use if B[i]["label"]==err and A[i]["label"]!=err)
    c = sum(1 for i in use if B[i]["label"]!=err and A[i]["label"]==err)
    d = len(use)-a-b-c
    n = len(use)
    if n == 0: return
    rA, rB = (a+c)/n, (a+b)/n
    lo,hi = newcombe10(a,b,c,d); wl,wh = wald_paired(b,c,n)
    lines.append(f"### paired: {nameB} vs {nameA}   n={n}" + (f"  (excluded {drop} no-label)" if drop else ""))
    lines.append(f"    {nameA} {rA*100:5.1f}%   {nameB} {rB*100:5.1f}%   discordant b={b} c={c}")
    lines.append(f"    exact McNemar p = {binom_two_sided(b,c):.3g}")
    lines.append(f"    difference ({nameB} - {nameA}) = {(rB-rA)*100:+.1f} pts   "
                 f"Newcombe [{lo*100:+.1f}, {hi*100:+.1f}]   Wald [{wl*100:+.1f}, {wh*100:+.1f}]")
    lines.append("")

def build_report(judge=JUDGE, repeat=REPEAT, cells=None):
    cells = cells or CELLS
    runs = {}
    for c in cells:
        p = RUNS_DIR / f"{judge}_{c}_r{repeat}.jsonl"
        if p.exists(): runs[c] = load_run(p)
    lines = [f"# REPORT — {judge} (repeat {repeat})", ""]
    if not runs:
        return "\n".join(lines + [f"no run files found in {RUNS_DIR} for judge "
                                   f"'{judge}' repeat {repeat}"])
    r0 = next(iter(runs.values()))[0]
    lines += [f"model `{r0['model_id']}` · {r0['reasoning_setting']} · {r0['quantization']}",
              f"engine {r0['engine']} {r0['engine_version']} · gpu {r0['gpu']} · seed {r0['seed']}",
              f"sampling {r0['sampling']} · max_new_tokens {r0['max_new_tokens']}",
              f"repo commit `{r0['repo_commit']}` · prereg commit `{r0['prereg_commit']}`", ""]
    if r0.get("engine_fallback_reason"):
        lines += [f"> vLLM fallback in effect: {r0['engine_fallback_reason'][:200]}", ""]
    lines.append("## Per cell"); lines.append("")
    for c in cells:
        if c in runs: cell_block(runs[c], c, lines)
    lines.append("## Paired"); lines.append("")
    byid = {c: {r["problem_id"]: r for r in rows} for c, rows in runs.items()}
    for a, b in [("CA","D"), ("D","persuasive"), ("CA","persuasive"), ("H2","H3")]:
        if a in byid and b in byid: paired_block(byid[a], byid[b], a, b, lines)
    # cross-judge
    lines.append("## Cross-judge error rate (same cells)"); lines.append("")
    others = {"sonnet5": "sonnet5_{c}_r1.jsonl", "opus55": "opus55_{c}_subset50_r1.jsonl"}
    hdr = f"    {'cell':<20}{judge:>16}" + "".join(f"{k:>12}" for k in others)
    lines.append(hdr); lines.append("    " + "-"*(20+16+12*len(others)))
    for c in cells:
        row = f"    {c:<20}"
        for src_name, pat in [(judge, None), *others.items()]:
            if pat is None:
                rows = runs.get(c)
            else:
                p = RUNS_DIR / pat.format(c=c)
                rows = load_run(p) if p.exists() else None
            if not rows: row += f"{'-':>16}" if pat is None else f"{'-':>12}"; continue
            s = summarise(rows, ERR(c))
            cellstr = f"{s['rate']*100:.1f}% ({s['HONEST'] if ERR(c)=='HONEST' else s['HACKING']}/{s['labelled']})"
            row += f"{cellstr:>16}" if pat is None else f"{cellstr:>12}"
        lines.append(row)
    lines.append("")
    return "\n".join(lines)

REPORT = build_report()
print(REPORT)
(RUNS_DIR / f"REPORT_{JUDGE}.md").write_text(REPORT, encoding="utf-8")
print("\nwrote", RUNS_DIR / f"REPORT_{JUDGE}.md")

In [ ]:
#@title 12 · Standalone report over any set of run files
# Edit and run this on its own; it does not need the engine loaded.
REPORT_JUDGE  = JUDGE      #@param {type:"string"}
REPORT_REPEAT = REPEAT     #@param {type:"integer"}
REPORT_CELLS  = None       # None = the CONFIG list

print(build_report(REPORT_JUDGE, REPORT_REPEAT, REPORT_CELLS))

In [ ]:
#@title 13 · Append ledger rows (existing schema)
import csv, json
LEDGER_FIELDS = ["experiment_id","experiment_name","date","source_dataset","condition",
                 "label_pair","n_expected","n_calls","n_valid_labels","n_no_label",
                 "n_positive","n_negative","model","effort","max_output_tokens",
                 "stop_reasons","source_result_file","prereg_or_amendment_source"]
EXPERIMENT_ID_PREFIX = "W"     # W01, W02, ... for open-weight judges

def append_ledger(judge=JUDGE, repeat=REPEAT, cells=None, amendment="Amendment (open weights)"):
    cells = cells or CELLS
    assert LEDGER_CSV.exists(), f"{LEDGER_CSV} not staged on Drive"
    existing = list(csv.DictReader(open(LEDGER_CSV, newline="", encoding="utf-8")))
    have = {r["source_result_file"] for r in existing}
    used = [r["experiment_id"] for r in existing if r["experiment_id"].startswith(EXPERIMENT_ID_PREFIX)]
    nxt = max([int(x[1:]) for x in used] or [0]) + 1
    new = []
    for c in cells:
        p = RUNS_DIR / f"{judge}_{c}_r{repeat}.jsonl"
        if not p.exists(): continue
        rel = f"runs/v2/{p.name}"
        if rel in have: print("skip (already in ledger):", rel); continue
        rows = load_run(p); r0 = rows[0]
        from collections import Counter
        stops = Counter(r.get("stop_reason") or "none" for r in rows)
        new.append({
            "experiment_id": f"{EXPERIMENT_ID_PREFIX}{nxt:02d}",
            "experiment_name": f"v2 {c} natural ({r0['run_id']})",
            "date": (r0["timestamp"] or "")[:10],
            "source_dataset": f"{r0['input_file']} sha256:{r0['input_sha256']}",
            "condition": r0["prompt_variant"],
            "label_pair": f"{r0['positive_label']}/{r0['negative_label']}",
            "n_expected": len(rows), "n_calls": len(rows),
            "n_valid_labels": sum(1 for r in rows if not r["no_label"]),
            "n_no_label": sum(1 for r in rows if r["no_label"]),
            "n_positive": sum(1 for r in rows if r["label"]=="HACKING"),
            "n_negative": sum(1 for r in rows if r["label"]=="HONEST"),
            "model": r0["model_id"], "effort": r0["reasoning_setting"],
            "max_output_tokens": max(r["output_tokens"] or 0 for r in rows),
            "stop_reasons": ";".join(f"{k}:{v}" for k,v in sorted(stops.items())),
            "source_result_file": rel,
            "prereg_or_amendment_source": f"notes/v2_prereg.md @ {r0['prereg_commit'][:12]} ({amendment})",
        }); nxt += 1
    with open(LEDGER_CSV, "a", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=LEDGER_FIELDS)
        for rec in new:
            w.writerow(rec); print("appended:", rec["experiment_id"], rec["experiment_name"])
    if not new: print("nothing appended")

# append_ledger()   # uncomment to write

## Notes and known limits

- **The prompt assertion is the safety rail.** Every cell re-derives the Sonnet 5 prompt with
  `{LAB}="Anthropic"` and compares its sha256 to the value logged in
  `runs/v2/sonnet5_<cell>_r1.jsonl`. If the assembly has drifted for any reason the run stops
  before spending GPU time. The second half of the check proves the judge's prompt differs from
  the Anthropic one only by the `{LAB}` substitution.
- **`usd` is `None`** for open-weight runs; there is no per-token price. Cost accounting in
  `src/costing.py` is not used.
- **`reasoning_summary` is truncated to 2000 characters** to keep the log readable and match the
  Anthropic field, which holds an API-provided summary. The full chain of thought is kept
  verbatim in `raw_reasoning`, and the complete untouched generation in `raw_generation`.
- **Truncation is a no-label.** `stop_reason == "length"` with no `<label>` tag yields
  `label: null`, `no_label: true`, and is never resampled — the same rule the Anthropic runs use.
- **`gpt-oss-120b` needs an 80 GB GPU.** It is MXFP4 and will not load on T4 or L4.
- **Qwen has no official 4-bit release.** The official quantized build is
  `Qwen/Qwen3.8-27B-FP8`; this notebook loads bitsandbytes nf4 at load time as instructed, and
  records `quantization` per row so the two are never confused in analysis.
- **Sampling is stochastic.** Unlike the Anthropic runs these judges sample at the card's
  recommended thinking-mode settings, so repeats will differ. `SEED` is recorded per row.